In [94]:
# Imports
import pandas as pd
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os
import logging
import yaml
import click
import gzip, csv
from contextlib import closing
#from io import StringIO

In [ ]:
# Enable logging
logging.basicConfig(
    format="{asctime} - {levelname} - {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M",
    filename="logs_ingestion_pipeline.log"
)
#logger = logging.getLogger(__name__)
logging.error("Something went wrong")

2026-10-18 11:00 - ERROR - Something went wrong


In [4]:
# Check for existence of credentials, if not found, fail early
load_dotenv(override=True)

def require_env(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise EnvironmentError(f"Umgebungsvariable {key} fehlt")
    return val

In [5]:
# Database credentials
host = require_env("HOST")
dbname = require_env("DATABASE")
user = require_env("USER")
password = require_env("PASSWORD")

DSN = f"host={host} dbname={dbname} user={user} password={password}"

In [ ]:
# Read metadata: tables and column for creating datawarehouse
def read_data_contract(file_path="data_ingestion_contract.yaml") -> None:
    try:
        # Open and parse YAML
        with open(file_path, 'r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)

        # Ensure YAML data is list/dict for DataFrame conversion
        if isinstance(yaml_data, dict):
            # Convert dict to list of dicts for DataFrame
            yaml_data = [yaml_data]
        elif not isinstance(yaml_data, list):
            raise ValueError("YAML content must be a list or dict to convert to DataFrame.")

        return yaml_data
    except Exception as e:
        print(f"Unexpected error: {e}")


In [7]:
data_contract = read_data_contract()
print(data_contract)

[{'listings': ['estimated_occupancy_l365d', 'number_of_reviews_ly', 'hosts_time_as_user_months', 'property_type', 'host_name', 'listing_url', 'room_type', 'number_of_reviews', 'calendar_last_scraped', 'availability_60', 'host_since', 'accommodates', 'first_review', 'maximum_nights', 'amenities', 'description', 'neighborhood_overview', 'bathrooms', 'last_review', 'review_scores_cleanliness', 'availability_365', 'minimum_nights', 'review_scores_rating', 'latitude', 'calculated_host_listings_count_shared_rooms', 'maximum_minimum_nights', 'availability_90', 'availability_30', 'price', 'number_of_reviews_l30d', 'host_profile_url', 'bathrooms_text', 'minimum_nights_avg_ntm', 'host_thumbnail_url', 'has_availability', 'name', 'price_quote_checkin_date', 'review_scores_communication', 'host_location', 'calculated_host_listings_count', 'host_listings_count', 'host_identity_verified', 'hosts_time_as_host_months', 'source', 'host_id', 'neighbourhood', 'calculated_host_listings_count_private_rooms'

In [115]:
create_raw_schema_tables()

In [114]:
# Create tables and a schema for the raw data
def create_raw_schema_tables(DSN=DSN) -> None:
    try:
        with closing(psycopg2.connect(DSN)) as conn:
            with conn:
                with conn.cursor() as cur:
                    # Create schema
                    cur.execute("CREATE SCHEMA IF NOT EXISTS airbnb_raw_data AUTHORIZATION airbnb_berlin")
                    # Create tables
                    for table, columns in data_contract[0].items():
                        expected_columns = columns

                        if "data_cohort" not in expected_columns:
                            expected_columns.insert(0, "data_cohort")

                        column_def = sql.SQL(", ").join(sql.SQL("{} TEXT").format(
                            sql.Identifier(column)
                        ) for column in expected_columns)

                        query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({})"
                                        ).format(sql.Identifier("airbnb_raw_data",table), column_def)
                        cur.execute(query)

                        # Validate table existence
                        cur.execute("""
                        SELECT column_name FROM information_schema.columns
                        WHERE table_schema = %s AND table_name = %s
                        """, ("airbnb_raw_data", table))

                        actual_columns = {row[0] for row in cur.fetchall()}
                        missing = set(expected_columns) - actual_columns
                        if missing: raise ValueError(f"Table {table}: Missing columns {missing}")

                        logging.info("Validated table %s: %d columns",table,len(actual_columns))

                        actual_columns = {row[0] for row in cur.fetchall()}

            logging.info("Raw schema and table creation successful.")
    except Exception:
        logging.exception("Schema creation failed")
        raise


In [ ]:
# List existing data for ingestion
def find_existing_data(filepath: str = require_env("RAWDATAPATH"))-> dict:
    candidates = {}
    for dir in os.listdir(filepath):
        if dir.startswith("data_") and os.path.isdir(dir):
            candidates[dir] = []
            for file in os.listdir(filepath + "/" + dir):
                if file.startswith(("listings","reviews")) and file.endswith(".csv.gz"):
                    candidates[dir].append(file)

    return candidates

In [ ]:
# Detect schema drift
def read_header(folders_specified: tuple) -> list[str]:
    existing_data = find_existing_data()
    headers = {}

    for folder in existing_data:
        if folder in folders_specified:

            headers[folder] = {}

            for file in existing_data[folder]:
                with gzip.open(f"{folder}/{file}", "rt", encoding="utf-8", newline="") as f:
                    header = next(csv.reader(f))
                    headers[folder][file] = header

    return headers

def check_contract(folders_specified: tuple) -> None:
    data_contract = read_data_contract()
    unknown = []
    missing = []
    for folder in folders_specified:
        if folder in data_contract[0].keys():
            headers_folders_specified = read_header(folders_specified)
            print(f"headers_folders_specified: {headers_folders_specified}")
            unknown.append(set(headers_folders_specified) - data_contract[folder])
            missing.append(data_contract[folder] - set(headers_folders_specified))
    if unknown:
        raise ValueError(f"{table}: Unknown columns: {sorted(unknown)}")
        logger.error("Schema drift detected: unknown columns in table %s: %s", table, sorted(unknown))
    if missing:
        raise ValueError(f"{table}: Missing columns: {sorted(missing)}")
        logger.warn("Schema drift detected: missing columns in table %s: %s (become NULL)", table, sorted(missing))
    return unknown

In [117]:
x = check_contract(('data_26.06.26', 'data_25.09.23'))
print(x)

[]


In [116]:
load_file(('data_26.06.26',))

In [107]:
# Ingest data from local files into the tables via temporary table
def load_file(folders_for_ingestion: tuple) -> None:
    check_contract(folders_for_ingestion)
    existing_data = find_existing_data()

    headers = read_header(folders_for_ingestion)
    for folder in folders_for_ingestion:
        data_cohort = folder.removeprefix("data_")
        for file in existing_data[folder]:
            table = file.removesuffix(".csv.gz")
            header = headers[folder][file]
            column_definitions = sql.SQL(", ").join(sql.SQL("{} text").format(sql.Identifier(c)) for c in header)
            columns = sql.SQL(", ").join(map(sql.Identifier, header))
            logger.info("Starting ingestion %s/%s",table,file)
            try:
                with closing(psycopg2.connect(DSN)) as conn:
                    with conn.cursor() as cur:
                        # Temporary staging table
                        cur.execute(sql.SQL("CREATE TEMP TABLE tmp_load ({}) ON COMMIT DROP").format(column_definitions))
                        # Copy csv into staging
                        with gzip.open(f"{folder}/{file}", "rt", encoding="utf-8", newline="") as f:
                            cur.copy_expert("COPY tmp_load FROM STDIN WITH (FORMAT csv, HEADER true)", f)
                        # Validate row count
                        cur.execute("SELECT COUNT(*) FROM tmp_load")
                        staged_rows = cur.fetchone()[0]
                        if staged_rows == 0:
                             raise ValueError(f"No data found in {file}")
                        # Delete existing cohort
                        cur.execute(sql.SQL(
                            "DELETE FROM {} WHERE data_cohort = %s").format(sql.Identifier("airbnb_raw_data", table)), (data_cohort,)
                        )
                        # Insert new cohort into raw table
                        cur.execute(sql.SQL(
                                "INSERT INTO airbnb_raw_data.{t} (data_cohort, {c}) SELECT %s, {c} FROM tmp_load"
                        ).format(t=sql.Identifier(table), c=columns), (data_cohort,))
                        inserted_rows = cur.rowcount
                logger.info(
                    "Ingestion successful: table=%s, cohort=%s, staged=%d, inserted=%d",
                    table, data_cohort, staged_rows, inserted_rows
                )
            except Exception:
                logger.exception("Ingestion failed: table=%s, file=%s", table, file)
                raise

In [118]:
def run_pipeline(folders_for_ingestion):
    create_raw_schema_tables()
    check_contract(folders_for_ingestion)
    load_file(folders_for_ingestion)

In [119]:
run_pipeline(('data_26.06.26',))

In [ ]:
with closing(psycopg2.connect(DSN)) as conn:
    if conn:
        conn.close()